In [11]:
!pip install ipywidgets

import os
import shutil
import torch
from transformers import pipeline
import whisper
import librosa
import soundfile as sf
import yt_dlp
import time

In [12]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn",device=0 if torch.cuda.is_available() else -1)

In [13]:
def youtube_to_mp3(youtube_url: str, output_dir: str = "downloads") -> str:
    ydl_config = {
        "format": "bestaudio/best",
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "320",
        }],
        "outtmpl": os.path.join(output_dir, "%(title)s.%(ext)s"),
        "verbose": False, 
    }
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print(f"Downloading audio from {youtube_url}")
    
    try:
        with yt_dlp.YoutubeDL(ydl_config) as ydl:
            info = ydl.extract_info(youtube_url, download=True)
            filename = ydl.prepare_filename(info).rsplit(".", 1)[0] + ".mp3"
            return filename
    except Exception as e:
        print(f"Download error: {e}")
        return ""


In [14]:
def chunk_audio(filename: str, segment_length: int, output_dir: str) -> list:
    print(f"Chunking audio into {segment_length} second segments...")
    
    
    if not os.path.isdir(output_dir):
        os.makedirs(output_dir)
    
    audio, sr = librosa.load(filename, sr=16000)
    
    duration = librosa.get_duration(y=audio, sr=sr)
    num_segments = int(duration / segment_length)
    if duration % segment_length != 0:
        num_segments += 1
    
    chunked_files = []
    for i in range(num_segments):
        start = i * segment_length * sr
        end = min((i + 1) * segment_length * sr, len(audio))
        segment = audio[start:end]
        output_file = os.path.join(output_dir, f"segment_{i:03d}.mp3")
        sf.write(output_file, segment, sr)
        chunked_files.append(output_file)
    
    return chunked_files

In [15]:
def transcribe_audio(audio_files: list, output_dir: str = None, model_name="base") -> list:
    print("Transcribing audio...")

    model = whisper.load_model(model_name)
    transcripts = []
    
    transcripts_dir = os.path.join(output_dir, "transcripts") if output_dir else None
    if transcripts_dir:
        os.makedirs(transcripts_dir, exist_ok=True)
        
    main_transcript_file = os.path.join(output_dir, "full_transcript.txt") if output_dir else None
    
    for ind, audio_file in enumerate(audio_files):
        try:
            print(f"Transcribing segment {ind + 1}/{len(audio_files)}")
            result = model.transcribe(audio_file)
            transcript_text = result["text"]
            transcripts.append(transcript_text)
            
            if transcripts_dir:
                segment_file = os.path.join(transcripts_dir, f"segment_{ind+1:03d}_transcript.txt")
                with open(segment_file, "w", encoding="utf-8") as f:
                    f.write(f"Segment {ind+1} Transcript:")
                    f.write("\n\n\n")
                    f.write("="* 67 + "\n\n")
                    f.write(transcript_text)
                    f.write("\n\n")
                    
            if main_transcript_file:
                with open(main_transcript_file, "a", encoding="utf-8") as f:
                    f.write(f"\nSegment {ind+1}:\n\n\n")
                    f.write("="* 50 + "\n")
                    f.write(transcript_text)
                    f.write("\n\n")
                    
        except Exception as e:
            print(f"Error transcribing segment {ind + 1}: {e}")
            transcripts.append("")
            
            if transcripts_dir:
                error_file = os.path.join(transcripts_dir, f"segment_{ind+1:03d}_error.txt")
                with open(error_file, "w", encoding="utf-8") as f:
                    f.write(f"Error transcribing segment {ind+1}: {str(e)}")
    
    return transcripts

In [16]:
def init_summarizer():
    device = 0 if torch.cuda.is_available() else -1  
    print(f"Using device: {'GPU' if device == 0 else 'CPU'}")  
    
    try:
        return pipeline(
            "summarization",
            model="facebook/bart-large-cnn",
            framework="pt",
            device=device  
        )
    except Exception as e:
        print(f"Error initializing summarizer with large model: {e}")
        return pipeline(
            "summarization",
            model="facebook/bart-base",
            framework="pt",
            device=device  
        )

In [17]:
def summarize(chunks: list[str], max_length: int = 150, min_length: int = 30) -> list:
    
    print("Generating summary...")
    
    summarizer = init_summarizer()
    summaries = []
    
    for ind, chunk in enumerate(chunks):
        if not chunk.strip():  
            continue
            
        try:
            print(f"Summarizing chunk {ind + 1}/{len(chunks)}")
            
            if len(chunk) > 1024:
                chunk = chunk[:1024]
            
            summary = summarizer(chunk, 
                               max_length=max_length, 
                               min_length=min_length, 
                               do_sample=False)
            summaries.append(summary[0]['summary_text'])
        except Exception as e:
            print(f"Error summarizing chunk {ind + 1}: {e}")
            summaries.append(f"[Summary failed for chunk {ind + 1}]")
    
    return summaries

In [18]:
def summarize_youtube_video(youtube_url: str, output_dir: str) -> tuple:
    try:
        os.makedirs(output_dir, exist_ok=True)
        raw_audio_dir = os.path.join(output_dir, "raw_audio")
        chunks_dir = os.path.join(output_dir, "chunks")
        os.makedirs(raw_audio_dir, exist_ok=True)
        os.makedirs(chunks_dir, exist_ok=True)

        full_transcript_file = os.path.join(output_dir, "full_transcript.txt")
        if os.path.exists(full_transcript_file):
            os.remove(full_transcript_file)

        print("Downloading audio...")
        audio_filename = youtube_to_mp3(youtube_url, raw_audio_dir)
        if not audio_filename:
            raise Exception("Failed to download audio")


        print("Chunking audio...")
        segment_length = 10 * 60
        chunked_files = chunk_audio(audio_filename, segment_length, chunks_dir)
        
        print("Transcribing audio...")
        transcripts = transcribe_audio(chunked_files, output_dir)
        
        print("Generating detailed summary...")
        long_summaries = summarize(transcripts)
        long_summary = "\n\n".join(long_summaries)
        
        print("Generating TL;DR summary...")
        short_summary = summarize([long_summary], max_length=32, min_length=30)[0]
        
        summary_file = os.path.join(output_dir, "summary.txt")
        with open(summary_file, "w", encoding="utf-8") as f:
            f.write("Long Summary:\n")
            f.write("="* 50 + "\n\n")
            f.write(long_summary)
            f.write("\n\nShort Summary:\n")
            f.write("="* 50 + "\n\n")
            f.write(short_summary)
    
        return long_summary, short_summary

    except Exception as e:
        print(f"Error in summarization process: {e}")
        return None, None

In [19]:
def main():
    print("="*67)
    youtube_url = input("Enter YouTube URL: ")
    outputs_dir = os.path.join(os.getcwd(), "youtube_summaries", 
                              f"summary_{time.strftime('%Y%m%d_%H%M%S')}")
    
    print("\nStarting video summarization process...")
    print(f"Output directory: {outputs_dir}")
    
    long_summary, short_summary = summarize_youtube_video(youtube_url, outputs_dir)
    
    if long_summary and short_summary:
        print("\nProcessing completed successfully!")
        print("\nSummary:")
        print("=" * 67)
        print(long_summary)
    else:
        print("\nFailed to generate summary.")

if __name__ == "__main__":
    main()


Starting video summarization process...
Output directory: /Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/Y2S/youtube_summaries/summary_20241026_235830
[youtube] Extracting URL: https://www.youtube.com/watch?v=Hu4Yvq-g7_Y
[youtube] Hu4Yvq-g7_Y: Downloading webpage
[youtube] Hu4Yvq-g7_Y: Downloading ios player API JSON
[youtube] Hu4Yvq-g7_Y: Downloading mweb player API JSON
[youtube] Hu4Yvq-g7_Y: Downloading m3u8 information
[info] Hu4Yvq-g7_Y: Downloading 1 format(s): 251-2
[download] Destination: /Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/Y2S/youtube_summaries/summary_20241026_235830/raw_audio/How to Get Your Brain to Focus ｜ Chris Bailey ｜ TEDxManchester.webm
[download] 100% of   10.63MiB in 00:00:03 at 3.19MiB/s   
[ExtractAudio] Destination: /Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/Y2S/youtube_summaries/summary_20241026_235830/raw_audio/How to Get Your Brain to Focus ｜ Chris Bailey ｜ TEDxManchester.mp3
Deleting o

/Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/venv/lib/python3.12/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  c

Transcribing segment 1/2


/Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/venv/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcribing segment 2/2


/Users/kousthubhngowda/Documents/PES/SEM 5/Machine Learning/ML-LAB/venv/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Generating detailed summary...
Generating summary...
Using device: CPU
Summarizing chunk 1/2
Summarizing chunk 2/2
Generating TL;DR summary...
Generating summary...
Using device: CPU
Summarizing chunk 1/1

Processing completed successfully!

Summary:
Our minds wander to think about the future more than the past and the present combined. This is called our mind's prospective bias and it occurs when
